## Imports

In [1]:
import kagglehub
import os
import torch
import torchvision as TV
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision.transforms.v2 import ToImage,ToDtype,Resize
from PIL import Image
from torch.utils.data import Dataset
from diffusers import DDPMPipeline
from tqdm import tqdm

device = "cuda"

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Load model and test pretrained output

In [ ]:
image_pipe = DDPMPipeline.from_pretrained("google/ddpm-celebahq-256")
image_pipe.to(device)

images = image_pipe().images
images[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--google--ddpm-celebahq-256/snapshots/cd5c944777ea2668051904ead6cc120739b86c4d: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--google--ddpm-celebahq-256/snapshots/cd5c944777ea2668051904ead6cc120739b86c4d.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


  0%|          | 0/1000 [00:00<?, ?it/s]

## Load the brain MRI dataset from kaggle

In [4]:
class BrainData(Dataset):
    def __init__(self, root, transform):
        self.root = root
        self.transform = transform
        self.samples = []

        # Use both training and testing folders
        train_dir = os.path.join(root, "Training")
        test_dir = os.path.join(root, "Testing")
        '''
        dirpath = os.path.join(train_dir, "glioma")
        for fname in os.listdir(dirpath):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                path = os.path.join(dirpath, fname)
                self.samples.append(path)

        #self.samples = self.samples[:100]
        '''
        for dirname in os.listdir(train_dir):
            if dirname.lower() == "notumor":
                continue
            dirpath = os.path.join(train_dir, dirname)
            for fname in os.listdir(dirpath):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    path = os.path.join(dirpath, fname)
                    self.samples.append(path)

        for dirname in os.listdir(test_dir):
            if dirname.lower() == "notumor":
                continue
            dirpath = os.path.join(test_dir, dirname)
            for fname in os.listdir(dirpath):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    path = os.path.join(dirpath, fname)
                    self.samples.append(path)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        imgpath = self.samples[idx]
        # Convert to Grayscale (L) then back to RGB to keep 3 channels to reduce complexity
        image = Image.open(imgpath).convert("L").convert("RGB")
        image = self.transform(image)
        return image, 0

image_size = 256
transforms = TV.transforms.Compose([
    Resize([image_size, image_size]),
    ToImage(),
    ToDtype(torch.float32, scale=True),
    TV.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
data = BrainData(root=path, transform=transforms)

print(f"Number of training samples: {len(data)}")


Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Number of training samples: 5400


## Training loop

In [ ]:
# Training setup
batch_size = 16
num_epochs = 12
lr = 1e-5
grad_accumulation_steps = 2

train_dataloader = torch.utils.data.DataLoader(
    data, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4
)

optimizer = torch.optim.AdamW(image_pipe.unet.parameters(), lr=lr)

losses = []

image_pipe.unet.train()
for epoch in range(num_epochs):
    for step, (x, _) in tqdm(enumerate(train_dataloader), total=len(train_dataloader)):
        x = x.to(device)
        # Sample noise to add to the images
        noise = torch.randn(x.shape).to(x.device)
        bs = x.shape[0]

        # Sample a random timestep for each image
        timesteps = torch.randint(
            0,
            image_pipe.scheduler.config.num_train_timesteps,
            (bs,),
            device=device,
        ).long()

        # Add noise to the clean images according to the noise magnitude at each timestep
        # (this is the forward diffusion process)
        noisy_images = image_pipe.scheduler.add_noise(x, noise, timesteps)

        # Get the model prediction for the noise
        noise_pred = image_pipe.unet(noisy_images, timesteps, return_dict=False)[0]

        # Compare the prediction with the actual noise:
        loss = F.mse_loss(
            noise_pred, noise
        )

        losses.append(loss.item())
        loss = loss / grad_accumulation_steps
        loss.backward()

        # Gradient accumulation:
        if (step + 1) % grad_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

    print(
        f"Epoch {epoch+1} average loss: {sum(losses[-len(train_dataloader):])/len(train_dataloader)}"
    )

    # Test training progress every epoch and save
    image_pipe.unet.eval()
    with torch.no_grad():
        generated_images = image_pipe(batch_size=4).images

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        for i, img in enumerate(generated_images):
            axes[i].imshow(img)
            axes[i].axis("off")

        plt.suptitle(f"Epoch {epoch+1} Outputs")
        plt.show()

    image_pipe.save_pretrained(f"checkpoint_epoch_{epoch+1}")

    image_pipe.unet.train()

# Plot the loss curve:
plt.figure(figsize=(8, 5))
plt.plot(losses, label = "Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()

## Load the best model and generate the synthetic dataset

In [ ]:

checkpoint_path = "checkpoint_epoch_10"
output_dir = "generated_images"
os.makedirs(output_dir, exist_ok=True)

pipe = DDPMPipeline.from_pretrained(checkpoint_path)
pipe.to(device)

num_images = 800
batch_size = 16

with torch.no_grad():
    for i in range(0, num_images, batch_size):
        batch_output = pipe(batch_size=batch_size).images

        for j, image in enumerate(batch_output):
            img_idx = i + j
            image.save(os.path.join(output_dir, f"mri_batch_{img_idx}.png"))

In [ ]:
from google.colab import files
files.download('generated_images.zip')

# Classification Module (Resnet)

In [ ]:
import cv2
import numpy as np
from tensorflow import Tensor
from tensorflow.keras.layers import Input, Conv2D, ReLU, BatchNormalization,\
                                    Add, AveragePooling2D, Flatten, Dense
from tensorflow.keras.models import Model
from keras.layers import Activation, Dropout, BatchNormalization, Dense
from keras.metrics import categorical_crossentropy
from keras.callbacks import EarlyStopping
import pandas as pds
import random
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models

from sklearn.model_selection import train_test_split


In [ ]:
output_dir = "generated_images"
#label = ['glioma', 'meningioma', 'pituitary', 'notumor']
label = ['notumor', 'tumor']

def read_image(img_path):
    img = cv2.imread(img_path)
    img = np.float32(cv2.resize(img, (256, 256)))
    img = img/255
    return img

def load_images(data_path):
  images = []
  for image_path in os.listdir(data_path):
      path = os.path.join(data_path,image_path)
      try:
          images.append(read_image(path))
      except: continue
  return np.array(images)

def combine_datasets(dataset1, dataset2, classes):
  dataset = np.concatenate((dataset1, dataset2), axis = 0)
  y1 = [classes[0]]*np.size(dataset1, axis = 0)
  y2 = [classes[1]]*np.size(dataset2, axis = 0)
  y = np.concatenate((np.array(y1), np.array(y2)))
  return dataset, y



In [ ]:
current_path = path
output_dir = "generated_images"

dirpath = os.path.join(current_path, "Testing/glioma")
raw_image = load_images(dirpath)

dirpath = os.path.join(current_path, "Training/glioma")
tumor_img_train = load_images(dirpath)


train_path_notumor = os.path.join(current_path, "Training/notumor")
val_path_notumor = os.path.join(current_path, "Testing/notumor")

train_notumor = load_images(train_path_notumor)
test_notumor = load_images(val_path_notumor)

gen_img = load_images(output_dir)
train_tumor = np.concatenate((gen_img, tumor_img_train[:100]), axis=0)
#train_tumor, test_tumor = train_test_split(datasets_tumor, test_size=0.2)
test_tumor = raw_image

print(test_tumor.shape)
print(train_tumor.shape)
print(test_notumor.shape)
print(train_notumor.shape)

x_train, y_train = combine_datasets(train_notumor, train_tumor, label)
x_test, y_test = combine_datasets(test_notumor, test_tumor, label)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def dataloader(x, y, batch_size=32, shuffle=True):
    label_map = {'notumor': 0, 'tumor': 1}
    y_int = np.array([label_map[l] for l in y])

    x_tensor = torch.tensor(x).permute(0, 3, 1, 2).float()
    y_tensor = torch.tensor(y_int).long()

    dataset = TensorDataset(x_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def get_resnet_model(num_classes=2):
    model = models.resnet18(weights='IMAGENET1K_V1')

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model

def train_model(epochs, train_loader, generated):
    model = get_resnet_model().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        checkpoint = {
          'epoch': epoch,
          'model_state_dict': model.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'loss': running_loss,
        }
        if generated:
          torch.save(checkpoint, 'tumor_checkpoint_generated.pth')
        else:
          torch.save(checkpoint, 'tumor_checkpoint_raw.pth')

        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.8f}")

# Training with Generated Image

In [ ]:
train_loader = dataloader(x_train, y_train)
test_loader = dataloader(x_test, y_test)

train_model(100, train_loader, True)


In [ ]:
model = get_resnet_model()
checkpoint = torch.load('tumor_checkpoint_generated.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model.to(device)

In [ ]:
def test_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            '''
            print(predicted)
            print(labels)

            if predicted != labels:
              plt.imshow(inputs)
              plt.title(f"Displayed Image - Predicted:{predicted}/Correct;{labels}")
              plt.axis('off')
              plt.show()
            '''
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    calculate_metrics(y_true, y_pred)
    accuracy = 100 * correct / total
    print(f'Test Accuracy: {accuracy:.2f}%')
    return all_labels, all_preds

def calculate_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    accuracy = (tp + tn) / len(y_true)

    print(f"True Positives:  {tp}")
    print(f"True Negatives:  {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print("-" * 20)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"Precision: {precision:.4f}")



In [ ]:
all_labels , all_preds = test_model(model, test_loader, device)
#calculate_metrics(all_labels, all_preds)

# Training with Raw Images

In [ ]:
current_path = path
output_dir = "generated_images"


dirpath = os.path.join(current_path, "Testing/glioma")
test_tumor = load_images(dirpath)


dirpath = os.path.join(current_path, "Training/glioma")
train_tumor = load_images(dirpath)
train_tumor = train_tumor[:100]

train_path_notumor = os.path.join(current_path, "Training/notumor")
val_path_notumor = os.path.join(current_path, "Testing/notumor")

train_notumor = load_images(train_path_notumor)
test_notumor = load_images(val_path_notumor)


print(test_tumor.shape)
print(train_tumor.shape)
print(test_notumor.shape)
print(train_notumor.shape)

x_train, y_train = combine_datasets(train_notumor, train_tumor, label)
x_test, y_test = combine_datasets(test_notumor, test_tumor, label)

In [ ]:
train_loader = dataloader(x_train, y_train)
test_loader = dataloader(x_test, y_test)

train_model(100, train_loader, False)

In [ ]:
model = get_resnet_model()
checkpoint = torch.load('tumor_checkpoint_raw.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model.to(device)

all_labels , all_preds = test_model(model, test_loader, device)
#calculate_metrics(all_labels, all_preds)